## 1.5 Load and Merge English Data

In [1]:
import pandas as pd
from pprint import pprint
from mlds import data_loader
from mlds.utils.converter import simple_majority_vote
# disable warnings
import warnings
warnings.filterwarnings('ignore')

entity_manager = data_loader.EntityDataManager(data_folder="data/json")
intent_manager = data_loader.IntentDataManager(
    data_folder="data/UtteranceGen/practice"
)

english_entity = entity_manager.load_english_data(reviewed=True, merged=True)
english_intent = intent_manager.load_english_data()


amh
['amh', 'Hana', 'Bereket T', 'Bereket A']	ewe
['ewe', 'ALASSE', 'KLOVE', 'APELETE']	hau
['hau', 'Ruqayya', 'Saminu', 'Yusuf']	ibo
['ibo', 'Adaeze', 'Winnie', 'Tochukwu']	kin
['Happy', 'Jean', 'Emile', 'Melance']	lin
['lin', 'Christian', 'Ebenya', 'Pacifique']	lug
['lug', 'Ivan', 'Ibra', 'Deborah']	orm
['orm', 'Maraf Mengesha', 'Irandufa Indebu', 'Irandufa and Maraf']	sna
['sna', 'Mr Mupini', 'Mr Zenda', 'Mr Mhizha']	sot
['sot', 'Jan1', 'Madibe', 'Refilwe']	swa
['swa', 'Iddy', 'Nelson', 'Pauline']	twi
['twi', 'Bernard', 'Godwin', 'Stephen']	wol
['wol', 'Fadel', 'Khady', 'Penda']	xho
['raw', 'DIKO', 'SILO', 'NYAKAMBI']	yor
['yor', 'Tolulope', 'Ganiyat', 'Johnson']	zul
['zul', 'Babalo', 'Brilliant', 'Neo']	

In [4]:
english_intent.drop(columns=["text"], inplace=True)
english_intent.rename(columns={"English translation": "text", "generated": "raw"}, inplace=True)
english_intent

,split,domain,intent,raw,text,language
0,train,travel,translate,"""አውቶቢሱን መጠቀም አፈልጋለው"" በኦሮምኛ እንዴት ነው ምለው?","how do I say ""I want to use the bus"" in Oromo ...",amh
1,train,banking,transfer,50000 ብር ከንግድ ባንክ አካውንቴ ወደሲቢኢ ብር ልታስተላልፍልኝ ትችላለህ?,Can you transfer 50000 birr from my commercial...,amh
2,train,utility,time,በአዲስ አበባ ስንት ሰአት ነው?,what time is it in Addis Ababa,amh
3,train,home,shopping_list_update,የአስቤዛ ዝርዝር ላይ ጨምርበት,Add teff to the groceries list.,amh
4,train,kitchen_and_dining,restaurant_reservation,ቶቶት ክትፎ ቤት እራት ቦታ ልቲዝልኝ ትችላለህ?,Can you reserve a dinner place for me at Totot...,amh
...,...,...,...,...,...,...
35,train,home,update_playlist,ngicela wengeze i-albhamu kaBritney Spears - i...,please add Britney Spears - Baby One More Time...,zul
36,train,kitchen_and_dining,cancel_reservation,ngiyikhansela kanjani indawo yami engiyibhukil...,how do I cancel my reservation at The Little I...,zul
37,train,travel,car_rental,kubiza malini ukuqasha iHyundai i20 kwaCeza?,how much does it cost to rent a Hyundai i20 in...,zul
38,train,kitchen_and_dining,meal_suggestion,phakamisa indawo yokudlela ye-vegan eWestville,suggest a vegan restaurant in Westville,zul


In [5]:
def to_entity_spans(data):
    result = []
    for annotation in data['annotations']:
        if annotation['completed_by']['id'] == 'MAJORITY_VOTE':
            for item in annotation['result']:
                start = item['value']['start']
                end = item['value']['end']
                label = item['value']['labels'][0]
                result.append(f"{start}:{end}:SL:{label}")

    return ','.join(sorted(result, key=lambda x: int(x.split(':')[0])))

def merge_entity_spans(text_a, spana, text_b, spanb, joint_char=""):
    # remember to add a space between the two texts
    # the start and end index of the second text should be updated
    l = []
    for item in spanb.split(","):
        if item == "":
            continue
        start, end, _, label = item.split(":")
        l.append( f"{int(start) + len(text_a) + len(joint_char)}:{int(end) + len(text_a) + len(joint_char)}:SL:{label}")
    spanb = ",".join(l)
    if spana:
        spana += "," + spanb
    return spanb
    
def to_logical_form(text, intent, spans):
    # [IN:GET_MESSAGE [SL:CONTACT Angelika Kratzer ] [SL:TYPE_CONTENT video ] [SL:RECIPIENT me ] ]
    intent = intent.replace(" ", "_")
    ss = spans.split(",")
    # if ss == [""]:
    #     return ""
    result = f"[IN:{intent} "
    for item in ss:
        if item == "":
            continue
        start, end, _, label = item.split(":")
        result += f"[SL:{label} {text[int(start):int(end)]}] "
    result += "]"
    return result


def merge_entity_intent(lan, lan_entity, lan_intent):
    print(f"[[[ Start {lan}")
    lan_entity = simple_majority_vote(lan_entity)

    lan_entity = pd.DataFrame([{
            # "text": item["data"]["text"],
            "text4match": item["data"]["text"].replace('"', '').lower(),
            "spans": to_entity_spans(item),
            # "changed": item["changed"],
        } for item in lan_entity])
    # lan_entity.drop(columns=["text"], inplace=True)
    # lan_entity.sort_values("text4match").to_csv(f"data/output/{lan}_entity.csv", index=False)
    
    # lan_intent = intent_manager.load_english_data()
    lan_intent["text4match"] = lan_intent["text"].str.replace('"', '').str.lower()#.str.replace('"', '').str.strip("\n")
    # lan_intent.drop(columns=["text"], inplace=True)
    # lan_intent.sort_values("text4match").to_csv(f"data/output/{lan}_intent.csv", index=False)

    # lan_entity = lan_entity[lan_entity["changed"]]
    print(f"lan_entity: {len(lan_entity)}")
    print(f"lan_intent: {len(lan_intent)}")

    # lan_intent.sort_values(by="text4match", inplace=True)#.to_csv(f"data/output/{lan}_intent.csv", index=False)
    # lan_entity.sort_values(by="text4match", inplace=True)

    # combine the two dataframes
    print(len(set(lan_entity["text4match"].tolist()).intersection(set(lan_intent["text4match"].tolist()))))
    # drop the duplicates and nan
    def clean(df):
        df = df.drop_duplicates(subset="text4match")
        df = df.dropna(subset=["text4match"])
        return df
    lan_intent = clean(lan_intent)
    lan_entity = clean(lan_entity)
    print(f"lan_entity: {len(lan_entity)}")
    print(f"lan_intent: {len(lan_intent)}")
    df = pd.merge(lan_intent, lan_entity, on="text4match", how="inner")
    # If both key columns contain rows where the key is a null value, those rows will be matched against each other. This is different from usual SQL join behaviour and can lead to unexpected results.

    print(f"merged df: {len(df)}")
    df["logical_form"]  = df.apply(lambda x: to_logical_form(x["text"], x["intent"], x["spans"]), axis=1)
    df.drop(columns=["text4match", "split"], inplace=True)
 
    return df

langs = []
for lan in data_loader.LANGUAGES:
    langs.append(
    merge_entity_intent(lan, english_entity[lan], english_intent[english_intent["language"] == lan])
    )


merged = pd.concat(langs)
merged.to_csv("data/output/eng.csv", index=False)

[[[ Start amh
lan_entity: 120
lan_intent: 120
120
lan_entity: 120
lan_intent: 120
merged df: 120
[[[ Start ewe
lan_entity: 120
lan_intent: 120
120
lan_entity: 120
lan_intent: 120
merged df: 120
[[[ Start hau
lan_entity: 120
lan_intent: 120
120
lan_entity: 120
lan_intent: 120
merged df: 120
[[[ Start ibo
lan_entity: 120
lan_intent: 120
120
lan_entity: 120
lan_intent: 120
merged df: 120
[[[ Start kin
lan_entity: 120
lan_intent: 120
119
lan_entity: 120
lan_intent: 120
merged df: 119
[[[ Start lin
lan_entity: 120
lan_intent: 120
120
lan_entity: 120
lan_intent: 120
merged df: 120
[[[ Start lug
lan_entity: 93
lan_intent: 93
93
lan_entity: 93
lan_intent: 93
merged df: 93
[[[ Start orm
lan_entity: 120
lan_intent: 120
100
lan_entity: 102
lan_intent: 102
merged df: 100
[[[ Start sna
lan_entity: 93
lan_intent: 96
93
lan_entity: 93
lan_intent: 93
merged df: 93
[[[ Start sot
lan_entity: 120
lan_intent: 120
120
lan_entity: 120
lan_intent: 120
merged df: 120
[[[ Start swa
lan_entity: 120
lan_intent: 

In [6]:
merged

,domain,intent,raw,text,language,spans,logical_form
0,travel,translate,"""አውቶቢሱን መጠቀም አፈልጋለው"" በኦሮምኛ እንዴት ነው ምለው?","how do I say ""I want to use the bus"" in Oromo ...",amh,40:45:SL:LANGUAGE_NAME,[IN:translate [SL:LANGUAGE_NAME Oromo] ]
1,banking,transfer,50000 ብር ከንግድ ባንክ አካውንቴ ወደሲቢኢ ብር ልታስተላልፍልኝ ትችላለህ?,Can you transfer 50000 birr from my commercial...,amh,"17:27:SL:MONEY,36:51:SL:BANK_NAME,63:71:SL:PAY...",[IN:transfer [SL:MONEY 50000 birr] [SL:BANK_NA...
2,utility,time,በአዲስ አበባ ስንት ሰአት ነው?,what time is it in Addis Ababa,amh,19:30:SL:CITY_OR_PROVINCE,[IN:time [SL:CITY_OR_PROVINCE Addis Ababa] ]
3,home,shopping_list_update,የአስቤዛ ዝርዝር ላይ ጨምርበት,Add teff to the groceries list.,amh,4:8:SL:DISH_OR_FOOD,[IN:shopping_list_update [SL:DISH_OR_FOOD teff] ]
4,kitchen_and_dining,restaurant_reservation,ቶቶት ክትፎ ቤት እራት ቦታ ልቲዝልኝ ትችላለህ?,Can you reserve a dinner place for me at Totot...,amh,"18:24:SL:MEAL_PERIOD,41:58:SL:RESTAURANT_NAME",[IN:restaurant_reservation [SL:MEAL_PERIOD din...
...,...,...,...,...,...,...,...
95,home,update_playlist,ngicela wengeze i-albhamu kaBritney Spears - i...,please add Britney Spears - Baby One More Time...,zul,"11:25:SL:ARTIST_NAME,28:46:SL:SONG_NAME",[IN:update_playlist [SL:ARTIST_NAME Britney Sp...
96,kitchen_and_dining,cancel_reservation,ngiyikhansela kanjani indawo yami engiyibhukil...,how do I cancel my reservation at The Little I...,zul,34:61:SL:RESTAURANT_NAME,[IN:cancel_reservation [SL:RESTAURANT_NAME The...
97,travel,car_rental,kubiza malini ukuqasha iHyundai i20 kwaCeza?,how much does it cost to rent a Hyundai i20 in...,zul,47:51:SL:PLACE_NAME,[IN:car_rental [SL:PLACE_NAME Ceza] ]
98,kitchen_and_dining,meal_suggestion,phakamisa indawo yokudlela ye-vegan eWestville,suggest a vegan restaurant in Westville,zul,30:39:SL:PLACE_NAME,[IN:meal_suggestion [SL:PLACE_NAME Westville] ]


In [14]:
slot_manager = data_loader.SlotDataManager(data_folder="data/output")
eng = slot_manager.load_data("eng", "split")

Missing balance with {'sna', 'lug'}
Missing confirm_reservation with {'sna', 'lug'}
Missing freeze_account with {'sna', 'lug'}
Missing restaurant_reservation with {'sna', 'lug'}
Missing shopping_list_update with {'sna', 'lug'}
Missing time with {'sna', 'lug'}
Missing timezone with {'sna', 'lug'}
Missing transfer with {'sna', 'lug'}
Missing translate with {'sna', 'lug'}


In [16]:
# eng.columns
# Index(['domain', 'intent', 'raw', 'text', 'language', 'spans', 'logical_form',
#        'xtreme-up', 'source_language'],
#       dtype='object')
for k in ["train", "dev", "test"]:
    print(eng[k][['intent', 'spans']].groupby("intent").count()["spans"].sort_values(ascending=False).mean())

22.4
2.325
12.7


In [12]:
eng[['intent', 'spans']].groupby("intent").count()["spans"].sort_values(ascending=False).mean()

37.425